In [2]:
!pip install --upgrade pip
!pip install autogluon==1.1.0


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 25.3 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of thinc to determine which version is compatible with other requirements. This could take a while.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of openxlab to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multi

In [3]:
# ====================================================
# 0. Установка и импорт библиотек
# ====================================================

# Если ты запускаешь это в Colab / Kaggle, раскомментируй:
# !pip install -U pip
# !pip install -U autogluon

import os
import random
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

from autogluon.tabular import TabularPredictor

# ====================================================
# 1. Фиксация сидов (просидировать ноут)
# ====================================================

RANDOM_SEED = 42

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

seed_everything(RANDOM_SEED)



In [4]:
# ====================================================
# 2. Конфигурация путей и колонок (чтобы было удобно переиспользовать)
# ====================================================

# Папка с данными
DATA_DIR = "./"  # поменяй при необходимости

TRAIN_PATH = os.path.join(DATA_DIR, "/kaggle/input/f-dataset/train_dataset (1).parquet")
TEST_PATH = os.path.join(DATA_DIR, "/kaggle/input/f-dataset/test_dataset (1).parquet")
SAMPLE_SUB_PATH = os.path.join(DATA_DIR, "/kaggle/input/f-dataset/sample_submission (4).csv")

# Названия ID и таргета (легко поменять для другой задачи)
ID_COL = "ID"
TARGET_COL = "price_TARGET"

# # Папки с изображениями (в этом шаблоне мы их не используем,
# # но оставим, чтобы потом было проще расширить до мульти-модального решения)
# TRAIN_IMAGES_DIR = os.path.join(DATA_DIR, "train_images")
# TEST_IMAGES_DIR = os.path.join(DATA_DIR, "test_images")

# Путь для сохранения модели и сабмита
MODEL_DIR = "./autogluon_models"
SUBMISSION_PATH = "./submission.csv"

# ====================================================
# 3. Загрузка данных
# ====================================================

train_df = pd.read_parquet(TRAIN_PATH)
test_df = pd.read_parquet(TEST_PATH)
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Sample submission head:")
display(sample_sub.head())


Train shape: (70000, 35)
Test shape: (25000, 34)
Sample submission head:


,ID,target
0,8,100000
1,26,100000
2,33,100000
3,52,100000
4,119,100000


In [5]:
# ====================================================
# 4. Функция генерации фич (отдельный шаг)
#    — можно расширять под задачу, сейчас минимальный вариант
# ====================================================

def add_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    
    # 4.1. Дата закрытия объявления
    if "close_date" in df.columns:
        df["close_date"] = pd.to_datetime(df["close_date"], errors="coerce")
        df["close_year"] = df["close_date"].dt.year
        df["close_month"] = df["close_date"].dt.month
        df["close_dayofweek"] = df["close_date"].dt.dayofweek
    
    # 4.2. Пробег — логарифм
    if "mileage" in df.columns:
        df["mileage_log"] = np.log1p(df["mileage"])
    
    # 4.3. Координаты — округление (грубое кластеризование по региону)
    if "latitude" in df.columns and "longitude" in df.columns:
        df["lat_round1"] = df["latitude"].round(1)
        df["lon_round1"] = df["longitude"].round(1)
    
    return df

train_df = add_features(train_df)
test_df = add_features(test_df)


In [6]:
# ====================================================
# 5. Подготовка списков числовых и категориальных фич
#    + краткая обработка пропусков и категориальных данных
#    - категориальные: заполняем пропуски (и нули) модой
#    - числовые: пропуски средним
# ====================================================

# Колонки, которые точно не используем как признаки
exclude_cols = {ID_COL, TARGET_COL}

# Если какие-то технические колонки не нужны — добавь сюда руками:
# exclude_cols.update(["some_tech_column"])

feature_cols = [c for c in train_df.columns if c not in exclude_cols]

# Разделяем на числовые и категориальные
num_cols = []
cat_cols = []

for col in feature_cols:
    if pd.api.types.is_numeric_dtype(train_df[col]):
        num_cols.append(col)
    else:
        cat_cols.append(col)

print("Numeric features:", num_cols[:10], " ... total:", len(num_cols))
print("All categorical features:", cat_cols[:10], " ... total:", len(cat_cols))

# ---- отделяем колонки, в которых лежат списки ----
def is_list_column(series):
    return series.apply(lambda x: isinstance(x, list)).any()

list_cols = [c for c in cat_cols if is_list_column(train_df[c])]
simple_cat_cols = [c for c in cat_cols if c not in list_cols]

print("List-type categorical columns:", list_cols)
print("Simple categorical columns:", simple_cat_cols[:10])



import collections.abc
import numpy as np

# ---------------------------------------------------------
# Функция: колонка плохая, если содержит НЕ-скалярные объекты
# ---------------------------------------------------------
def is_bad_column(series):
    return series.apply(
        lambda x: not (
            isinstance(x, (str, int, float, bool)) 
            or x is None 
            or (isinstance(x, float) and np.isnan(x))
        )
    ).any()

bad_cols = [c for c in cat_cols if is_bad_column(train_df[c])]

print("Bad categorical columns (will drop):", bad_cols)

# дропаем
train_df = train_df.drop(columns=bad_cols)
test_df = test_df.drop(columns=bad_cols)

# обновляем simple_cat_cols
simple_cat_cols = [c for c in cat_cols if c not in bad_cols]
print("Final simple categorical cols:", simple_cat_cols)



# --- обработка категориальных: нули считаем пропусками ---
for col in simple_cat_cols:
    train_df[col] = train_df[col].replace({0: np.nan, "0": np.nan})
    test_df[col] = test_df[col].replace({0: np.nan, "0": np.nan})

# Импьютеры
num_imputer = SimpleImputer(strategy="mean")
cat_imputer = SimpleImputer(strategy="most_frequent")

# fit только на простые категориальные
if num_cols:
    num_imputer.fit(train_df[num_cols])
if simple_cat_cols:
    cat_imputer.fit(train_df[simple_cat_cols])

def apply_preprocessing(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    if num_cols:
        df[num_cols] = num_imputer.transform(df[num_cols])

    if simple_cat_cols:
        df[simple_cat_cols] = cat_imputer.transform(df[simple_cat_cols])
        for c in simple_cat_cols:
            df[c] = df[c].astype("category")

    # list_cols оставляем как есть — AutoGluon умеет их сам обрабатывать
    return df

train_proc = apply_preprocessing(train_df)
test_proc = apply_preprocessing(test_df)

print("Train processed shape:", train_proc.shape)
print("Test processed shape:", test_proc.shape)


Numeric features: ['doors_number', 'mileage', 'latitude', 'longitude', 'mileage_log', 'lat_round1', 'lon_round1']  ... total: 7
All categorical features: ['equipment', 'body_type', 'drive_type', 'engine_type', 'color', 'pts', 'audiosistema', 'diski', 'electropodemniki', 'fary']  ... total: 29
List-type categorical columns: []
Simple categorical columns: ['equipment', 'body_type', 'drive_type', 'engine_type', 'color', 'pts', 'audiosistema', 'diski', 'electropodemniki', 'fary']
Bad categorical columns (will drop): ['aktivnaya_bezopasnost_mult', 'audiosistema_mult', 'shini_i_diski_mult', 'electroprivod_mult', 'fary_mult', 'multimedia_navigacia_mult', 'obogrev_mult', 'pamyat_nastroek_mult', 'podushki_bezopasnosti_mult', 'pomosh_pri_vozhdenii_mult', 'protivoygonnaya_sistema_mult', 'salon_mult', 'upravlenie_klimatom_mult']
Final simple categorical cols: ['equipment', 'body_type', 'drive_type', 'engine_type', 'color', 'pts', 'audiosistema', 'diski', 'electropodemniki', 'fary', 'salon', 'uprav

In [7]:
# ====================================================
# 6. Тренировочный / валидационный сплит (для локальной оценки)
# ====================================================

train_data, val_data = train_test_split(
    train_proc,
    test_size=0.2,
    random_state=RANDOM_SEED
)

print("Train split shape:", train_data.shape)
print("Val split shape:", val_data.shape)


Train split shape: (56000, 25)
Val split shape: (14000, 25)


In [8]:
# ====================================================
# 7. Запуск AutoGluon "жесткий" (best_quality)
# ====================================================

# В качестве eval_metric возьмем MAE — она хорошо коррелирует с medianAPE,
# а саму medianAPE посчитаем вручную после обучения.

predictor = TabularPredictor(
    label=TARGET_COL,
    problem_type="regression",
    eval_metric="mean_absolute_error",
    path=MODEL_DIR,
    verbosity=2
).fit(
    train_data=train_data,
    presets="best_quality",
    time_limit=300,            # ← ограничиваем время
    num_bag_folds=5,
    num_stack_levels=1,
    dynamic_stacking=False,     # ← отключаем зависающее поведение
    ag_args_fit={"_disable_ray": True},   # ← отключаем ray
)



Presets specified: ['best_quality']
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=5, num_bag_sets=1
Beginning AutoGluon training ... Time limit = 300s
AutoGluon will save models to "./autogluon_models"
=================== System Info ===================
AutoGluon Version:  1.1.0
Python Version:     3.11.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP PREEMPT_DYNAMIC Sun Nov 10 10:07:59 UTC 2024
CPU Count:          4
Memory Avail:       29.75 GB / 31.35 GB (94.9%)
Disk Space Avail:   19.50 GB / 19.52 GB (99.9%)
Train Data Rows:    56000
Train Data Columns: 24
Label Column:       price_TARGET
Problem Type:       regression
Preprocessing data ...
Using Feature Generators to preprocess the data ...
Fitting AutoMLPipelineFeatureGenerator...
	Available Memory:                    30471.62 MB
	Train Data (Original)  Memory Usage: 4.42 MB (0.0% of available memory)
	Inferring data type of each feature based on column values. Set feat

In [9]:
# ====================================================
# 8. Оценка на валидации: считаем medianAPE
# ====================================================

def median_ape(y_true, y_pred):
    """
    medianAPE = median(|y_hat - y| / y)
    заодно защищаемся от деления на 0 (очень дешевые авто)
    """
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    eps = 1e-8
    ape = np.abs(y_pred - y_true) / np.clip(np.abs(y_true), eps, None)
    return np.median(ape)

val_pred = predictor.predict(val_data.drop(columns=[TARGET_COL]))
val_true = val_data[TARGET_COL].values

val_mape_median = median_ape(val_true, val_pred)
score = 1 / (1 + val_mape_median)

print(f"Validation medianAPE: {val_mape_median:.5f}")
print(f"Validation Score (1 / (1 + medianAPE)): {score:.5f}")


Validation medianAPE: 0.27432
Validation Score (1 / (1 + medianAPE)): 0.78473


In [10]:
# ====================================================
# 9. Вклад моделей AutoGluon: leaderboard
#    (видно, какие модели использовались и какова их вал.метрика)
# ====================================================

leaderboard = predictor.leaderboard(train_data, silent=True)
display(leaderboard)


,model,score_test,score_val,eval_metric,pred_time_test,pred_time_val,fit_time,pred_time_test_marginal,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,KNeighborsDist_BAG_L1,-0.000000,-582859.991620,mean_absolute_error,0.196389,0.191120,0.056176,0.196389,0.191120,0.056176,1,True,2
1,RandomForestMSE_BAG_L1,-100417.869447,-258337.015632,mean_absolute_error,2.738034,2.552576,87.701623,2.738034,2.552576,87.701623,1,True,5
2,LightGBM_BAG_L2,-155092.673145,-235029.864117,mean_absolute_error,36.350639,30.136658,189.252969,1.127950,0.801703,10.852105,2,True,9
3,RandomForestMSE_BAG_L2,-156579.067644,-237206.755747,mean_absolute_error,38.006895,32.116963,327.141448,2.784206,2.782007,148.740584,2,True,10
4,WeightedEnsemble_L3,-157387.045717,-232211.225215,mean_absolute_error,42.833842,35.091335,355.054686,0.004075,0.001251,0.168365,3,True,11
5,WeightedEnsemble_L2,-162697.724761,-235501.128745,mean_absolute_error,34.708501,28.808525,157.050186,0.002976,0.001339,0.122833,2,True,7
6,LightGBMXT_BAG_L2,-167175.338265,-234409.700258,mean_absolute_error,38.917611,31.506374,195.293633,3.694922,2.171419,16.892769,2,True,8
7,LightGBM_BAG_L1,-188622.137984,-241567.104095,mean_absolute_error,11.066276,9.436201,26.842260,11.066276,9.436201,26.842260,1,True,4
8,LightGBMXT_BAG_L1,-192776.531463,-244426.107685,mean_absolute_error,20.901214,16.818409,42.383470,20.901214,16.818409,42.383470,1,True,3
9,CatBoost_BAG_L1,-332238.677898,-338528.331453,mean_absolute_error,0.125267,0.150470,20.902074,0.125267,0.150470,20.902074,1,True,6


In [ ]:
# ====================================================
# 10. (Опционально) Важность признаков
# ====================================================

feature_importance = predictor.feature_importance(val_data)
display(feature_importance.head(30))


Computing feature importance via permutation shuffling for 24 features using 5000 rows with 5 shuffle sets...
	821.69s	= Expected runtime (164.34s per shuffle set)


In [ ]:
# ====================================================
# 11. Предсказание на тесте и сабмит
# ====================================================

# Предикт на тесте
test_pred = predictor.predict(test_proc)

# Создаем сабмит по шаблону
submission = sample_sub.copy()
# В шаблоне колонка называется "target", в задании — price_TARGET.
# Если у тебя другое имя, просто поправь здесь.
if "target" in submission.columns:
    submission["target"] = test_pred
else:
    # запасной вариант, если шаблон другой
    submission[TARGET_COL] = test_pred

submission.to_csv(SUBMISSION_PATH, index=False)
print(f"Submission saved to: {SUBMISSION_PATH}")
display(submission.head())
